# Seed07 Unlabeled Pseudo-Labeling

This notebook uses the saved `seed07` multiclass classifier to label the **unlabeled** rows in WM-811K.

Goal:
- run the `seed07` classifier on unlabeled WM-811K rows
- attach a pseudo label plus confidence for each row
- save a reusable pseudo-label table for later anomaly-threshold validation work
- optionally keep only a high-confidence subset for safer downstream use


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
            return candidate
    return None


def resolve_unique_input_path(patterns: str | list[str], description: str) -> Path:
    if isinstance(patterns, str):
        patterns = [patterns]
    matches_by_path: dict[str, Path] = {}
    for pattern in patterns:
        for match in sorted(Path("/kaggle/input").glob(pattern)):
            matches_by_path[str(match)] = match
    matches = sorted(matches_by_path.values())
    if len(matches) == 1:
        return matches[0]
    pattern_text = ", ".join(f"'/kaggle/input/{pattern}'" for pattern in patterns)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {description} with patterns {pattern_text}. "
            "Attach the Kaggle dataset first or set the path manually."
        )
    raise RuntimeError(
        f"Found multiple {description} matches: {matches}. "
        "Set the path manually to the correct mounted file."
    )


def resolve_first_existing_path(candidates: list[Path], description: str) -> Path:
    for path in candidates:
        if path.exists():
            return path
    candidate_text = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find {description}. Checked:\n{candidate_text}")


def run_logged(title: str, command: list[str], env: dict[str, str], cwd: Path) -> None:
    print(f"\n{'=' * 80}\n{title}\n{'=' * 80}")
    print("Command:", " ".join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)
    print(f"Completed: {title}\n")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if REPO_ROOT is None and Path('/kaggle/input').exists():
    bundle_source = None
    for patterns, description in [
        (['*/seed07_pseudolabel_bundle', '**/seed07_pseudolabel_bundle'], 'seed07 pseudo-label bundle'),
        (['*/classifier_all_80_10_10_bundle', '**/classifier_all_80_10_10_bundle'], 'classifier all-labeled bundle'),
    ]:
        try:
            bundle_source = resolve_unique_input_path(patterns, description)
            break
        except FileNotFoundError:
            continue
    if bundle_source is None:
        raise RuntimeError(
            'Could not locate a Kaggle bundle directory. Attach the exported classifier bundle dataset '
            'or run this notebook from the repository root.'
        )

    target = Path('/kaggle/working') / bundle_source.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(bundle_source, target)
    os.chdir(target)
    REPO_ROOT = find_repo_root(target.resolve())

if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("Repo root:", REPO_ROOT)
print("Working directory:", Path.cwd().resolve())


In [ ]:
base_data_config_path = REPO_ROOT / "configs/data/classifier/data_multiclass_all_80_10_10.toml"
RAW_PICKLE = None  # Optional manual override: Path('/kaggle/input/<dataset>/LSWMD.pkl')
if RAW_PICKLE is None:
    RAW_PICKLE = REPO_ROOT / "data/raw/LSWMD.pkl"
    if not RAW_PICKLE.exists():
        RAW_PICKLE = resolve_unique_input_path(["*/LSWMD.pkl", "**/LSWMD.pkl"], "raw WM-811K pickle")
else:
    RAW_PICKLE = Path(RAW_PICKLE)

runtime_dir = REPO_ROOT / "runtime_configs"
runtime_dir.mkdir(parents=True, exist_ok=True)
data_config_runtime = runtime_dir / "data_multiclass_all_80_10_10_pseudolabel.kaggle.toml"

config_text = base_data_config_path.read_text(encoding="utf-8")
config_text = config_text.replace(
    'raw_pickle = "data/raw/LSWMD.pkl"',
    f'raw_pickle = "{RAW_PICKLE.as_posix()}"',
)
data_config_runtime.write_text(config_text, encoding="utf-8")

checkpoint_path = resolve_first_existing_path(
    candidates=[
        REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed07/best_model.pt",
        REPO_ROOT / "outputs/_output_inspect/classifier_all_80_10_10_bundle/artifacts/multiclass_classifier_all_80_10_10_seed07/best_model.pt",
    ],
    description="seed07 checkpoint",
)

output_dir = REPO_ROOT / "artifacts/multiclass_classifier_all_80_10_10_seed07_pseudolabels"
raw_predictions_csv = output_dir / "unlabeled_predictions.seed07.raw.csv"
enriched_predictions_csv = output_dir / "unlabeled_predictions.seed07.pseudolabels.csv"
accepted_predictions_csv = output_dir / "unlabeled_predictions.seed07.accepted.csv"
summary_json = output_dir / "unlabeled_predictions.seed07.summary.json"

confidence_threshold = 0.98
limit = None  # Set an integer if you want a smaller smoke run.

output_dir.mkdir(parents=True, exist_ok=True)

print("Base data config:", base_data_config_path)
print("Runtime data config:", data_config_runtime)
print("Raw pickle:", RAW_PICKLE)
print("Checkpoint:", checkpoint_path)
print("Output dir:", output_dir)
print("Confidence threshold:", confidence_threshold)
print("Limit:", limit)


In [ ]:
predict_command = [
    sys.executable,
    "-u",
    str(REPO_ROOT / "scripts/classifier/predict_unlabeled_multiclass.py"),
    "--config",
    str(data_config_runtime),
    "--checkpoint",
    str(checkpoint_path),
    "--output-csv",
    str(raw_predictions_csv),
    "--min-confidence",
    str(confidence_threshold),
]
if limit is not None:
    predict_command.extend(["--limit", str(limit)])

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT / "src")
env["PYTHONUNBUFFERED"] = "1"

run_logged("Seed07 pseudo-label unlabeled WM-811K rows", predict_command, env=env, cwd=REPO_ROOT)


In [ ]:
predictions = pd.read_csv(raw_predictions_csv)

predictions = predictions.rename(
    columns={
        "predicted_label": "pseudo_label",
        "predicted_index": "pseudo_label_index",
        "confidence": "pseudo_label_confidence",
    }
)
predictions["pseudo_label_confidence_pct"] = predictions["pseudo_label_confidence"] * 100.0
predictions["second_choice_confidence_pct"] = predictions["second_choice_confidence"] * 100.0
predictions["confidence_margin"] = (
    predictions["pseudo_label_confidence"] - predictions["second_choice_confidence"]
)
predictions["predicted_is_defect"] = predictions["pseudo_label"] != "none"
predictions["checkpoint_name"] = checkpoint_path.parent.name

accepted_predictions = predictions[predictions["accepted_for_pseudo_label"]].copy()
accepted_predictions = accepted_predictions.sort_values(
    ["pseudo_label_confidence", "confidence_margin"],
    ascending=[False, False],
)

predictions.to_csv(enriched_predictions_csv, index=False)
accepted_predictions.to_csv(accepted_predictions_csv, index=False)

summary = {
    "checkpoint": str(checkpoint_path),
    "data_config_runtime": str(data_config_runtime),
    "source_predictions_csv": str(raw_predictions_csv),
    "enriched_predictions_csv": str(enriched_predictions_csv),
    "accepted_predictions_csv": str(accepted_predictions_csv),
    "confidence_threshold": float(confidence_threshold),
    "rows_scored": int(len(predictions)),
    "accepted_rows": int(len(accepted_predictions)),
    "accepted_fraction": float(accepted_predictions.shape[0] / max(1, predictions.shape[0])),
    "predicted_label_distribution": predictions["pseudo_label"].value_counts().sort_index().to_dict(),
    "accepted_label_distribution": accepted_predictions["pseudo_label"].value_counts().sort_index().to_dict(),
    "predicted_defect_fraction": float(predictions["predicted_is_defect"].mean()),
    "accepted_defect_fraction": float(accepted_predictions["predicted_is_defect"].mean()) if len(accepted_predictions) else 0.0,
    "mean_confidence": float(predictions["pseudo_label_confidence"].mean()),
    "mean_accepted_confidence": float(accepted_predictions["pseudo_label_confidence"].mean()) if len(accepted_predictions) else 0.0,
}
summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved enriched pseudo labels to:", enriched_predictions_csv)
print("Saved accepted pseudo labels to:", accepted_predictions_csv)
print("Saved summary to:", summary_json)


In [ ]:
display(predictions.head())
display(predictions[["pseudo_label_confidence", "pseudo_label_confidence_pct", "confidence_margin"]].describe())
display(predictions["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("count"))
display(accepted_predictions["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("accepted_count"))
print("Accepted fraction:", summary["accepted_fraction"])
print("Predicted defect fraction:", summary["predicted_defect_fraction"])
print("Accepted defect fraction:", summary["accepted_defect_fraction"])


In [ ]:
review_columns = [
    "raw_index",
    "pseudo_label",
    "pseudo_label_confidence",
    "pseudo_label_confidence_pct",
    "second_choice_label",
    "second_choice_confidence",
    "second_choice_confidence_pct",
    "confidence_margin",
    "accepted_for_pseudo_label",
    "predicted_is_defect",
]

print("Most confident pseudo labels")
display(predictions.sort_values("pseudo_label_confidence", ascending=False)[review_columns].head(20))

print("Least confident pseudo labels")
display(predictions.sort_values("pseudo_label_confidence", ascending=True)[review_columns].head(20))


In [ ]:
thresholds_to_review = [0.50, 0.75, confidence_threshold]
threshold_summary_rows = []

for threshold in thresholds_to_review:
    subset = predictions[predictions["pseudo_label_confidence"] >= threshold].copy()
    threshold_summary_rows.append(
        {
            "confidence_threshold": threshold,
            "confidence_threshold_pct": threshold * 100.0,
            "accepted_rows": int(len(subset)),
            "accepted_fraction": float(len(subset) / max(1, len(predictions))),
            "accepted_defect_rows": int(subset["predicted_is_defect"].sum()),
            "accepted_none_rows": int((~subset["predicted_is_defect"]).sum()),
        }
    )

threshold_summary = pd.DataFrame(threshold_summary_rows)
display(threshold_summary)

for threshold in thresholds_to_review:
    subset = predictions[predictions["pseudo_label_confidence"] >= threshold].copy()
    print(f"Pseudo labels with confidence >= {threshold:.0%}: {len(subset)} ({len(subset) / max(1, len(predictions)):.2%})")
    display(subset["pseudo_label"].value_counts().rename_axis("pseudo_label").to_frame("count"))


In [ ]:
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

try:
    import umap.umap_ as umap
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "UMAP plotting requires the 'umap-learn' package. Install it first, for example with `%pip install umap-learn`."
    ) from exc

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from wafer_defect.classification.data import DEFAULT_CLASS_NAMES, RawWaferInferenceDataset, prepare_supervised_dataframe
from wafer_defect.classification.models import WaferClassifier
from wafer_defect.config import load_toml


umap_min_confidence = 0.75
umap_max_labeled_per_class = 400
umap_max_pseudo_per_class = 800
umap_batch_size = 256
umap_n_neighbors = 30
umap_min_dist = 0.1
umap_metric = "cosine"
umap_random_seed = 7
umap_output_dir = output_dir / "umap_visualization"
umap_output_dir.mkdir(parents=True, exist_ok=True)


def stratified_sample(dataframe: pd.DataFrame, group_column: str, max_per_group: int, random_seed: int) -> pd.DataFrame:
    sampled_frames = []
    for _, group_df in dataframe.groupby(group_column, sort=True):
        if len(group_df) <= max_per_group:
            sampled_frames.append(group_df.copy())
        else:
            sampled_frames.append(group_df.sample(n=max_per_group, random_state=random_seed))
    return pd.concat(sampled_frames, ignore_index=True).reset_index(drop=True)


def extract_embedding(model: WaferClassifier, inputs: torch.Tensor) -> torch.Tensor:
    if hasattr(model, "extract_embedding"):
        return model.extract_embedding(inputs)

    x = model.stem(inputs)
    x = model.features(x)
    if getattr(model, "variant", "baseline") == "baseline":
        x = model.classifier[0](x)
        x = model.classifier[1](x)
        x = model.classifier[2](x)
        return model.classifier[3](x)

    avg_features = model.classifier.avgpool(x)
    max_features = model.classifier.maxpool(x)
    x = torch.cat([avg_features, max_features], dim=1)
    x = model.classifier.classifier[0](x)
    x = model.classifier.classifier[1](x)
    return model.classifier.classifier[2](x)


@torch.no_grad()
def collect_embeddings(model: WaferClassifier, dataframe: pd.DataFrame, image_size: int, batch_size: int, device: torch.device) -> torch.Tensor:
    dataset = RawWaferInferenceDataset(dataframe=dataframe, image_size=image_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    embedding_batches = []
    for inputs, _ in loader:
        embedding_batches.append(extract_embedding(model, inputs.to(device)).cpu())
    if not embedding_batches:
        raise ValueError("Could not extract embeddings because the dataframe is empty.")
    return torch.cat(embedding_batches, dim=0)


checkpoint = torch.load(checkpoint_path, map_location="cpu")
class_names = list(checkpoint.get("class_names", DEFAULT_CLASS_NAMES))
model_cfg = checkpoint["model_config"]
model = WaferClassifier(
    num_classes=len(class_names),
    base_channels=int(model_cfg["base_channels"]),
    hidden_dim=int(model_cfg["hidden_dim"]),
    dropout=float(model_cfg["dropout"]),
    variant=str(model_cfg.get("variant", "baseline")),
    block_dropout=float(model_cfg.get("block_dropout", 0.0)),
    se_reduction=int(model_cfg.get("se_reduction", 8)),
)
model.load_state_dict(checkpoint["model_state_dict"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

dataset_config = load_toml(data_config_runtime)["dataset"]
image_size = int(dataset_config["image_size"])
labeled_df, unlabeled_df = prepare_supervised_dataframe(raw_pickle=RAW_PICKLE, class_names=class_names)

labeled_umap_df = stratified_sample(
    labeled_df.rename(columns={"failure_type": "true_label"}),
    group_column="true_label",
    max_per_group=umap_max_labeled_per_class,
    random_seed=umap_random_seed,
)

pseudo_plot_predictions = predictions[predictions["pseudo_label_confidence"] >= umap_min_confidence].copy()
if pseudo_plot_predictions.empty:
    raise ValueError(f"No pseudo-labeled rows met the requested UMAP confidence threshold of {umap_min_confidence:.2f}.")
pseudo_plot_predictions = stratified_sample(
    pseudo_plot_predictions,
    group_column="pseudo_label",
    max_per_group=umap_max_pseudo_per_class,
    random_seed=umap_random_seed,
)

pseudo_umap_df = unlabeled_df[["raw_index", "waferMap"]].merge(
    pseudo_plot_predictions[
        [
            "raw_index",
            "pseudo_label",
            "pseudo_label_confidence",
            "second_choice_label",
            "second_choice_confidence",
            "accepted_for_pseudo_label",
        ]
    ],
    on="raw_index",
    how="inner",
    validate="one_to_one",
)

labeled_embeddings = collect_embeddings(
    model=model,
    dataframe=labeled_umap_df[["raw_index", "waferMap"]].copy(),
    image_size=image_size,
    batch_size=umap_batch_size,
    device=device,
)
pseudo_embeddings = collect_embeddings(
    model=model,
    dataframe=pseudo_umap_df[["raw_index", "waferMap"]].copy(),
    image_size=image_size,
    batch_size=umap_batch_size,
    device=device,
)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=umap_n_neighbors,
    min_dist=umap_min_dist,
    metric=umap_metric,
    random_state=umap_random_seed,
    transform_seed=umap_random_seed,
)
labeled_coords = reducer.fit_transform(labeled_embeddings.numpy())
pseudo_coords = reducer.transform(pseudo_embeddings.numpy())

labeled_points = labeled_umap_df[["raw_index", "true_label", "source_split"]].copy()
labeled_points["umap_x"] = labeled_coords[:, 0]
labeled_points["umap_y"] = labeled_coords[:, 1]

pseudo_points = pseudo_umap_df[
    [
        "raw_index",
        "pseudo_label",
        "pseudo_label_confidence",
        "second_choice_label",
        "second_choice_confidence",
        "accepted_for_pseudo_label",
    ]
].copy()
pseudo_points["umap_x"] = pseudo_coords[:, 0]
pseudo_points["umap_y"] = pseudo_coords[:, 1]

palette = {class_name: plt.get_cmap("tab10")(idx % 10) for idx, class_name in enumerate(class_names)}
fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

for class_name in class_names:
    label_mask = labeled_points["true_label"] == class_name
    if label_mask.any():
        axes[0].scatter(
            labeled_points.loc[label_mask, "umap_x"],
            labeled_points.loc[label_mask, "umap_y"],
            s=16,
            alpha=0.75,
            color=palette[class_name],
            label=class_name,
        )
axes[0].set_title("Labeled WM-811K Reference")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
axes[0].grid(alpha=0.2)
axes[0].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

axes[1].scatter(
    labeled_points["umap_x"],
    labeled_points["umap_y"],
    s=8,
    alpha=0.18,
    color="#9ca3af",
    label="Labeled reference",
)
for class_name in class_names:
    label_mask = pseudo_points["pseudo_label"] == class_name
    if label_mask.any():
        point_sizes = 12.0 + 26.0 * pseudo_points.loc[label_mask, "pseudo_label_confidence"].to_numpy()
        axes[1].scatter(
            pseudo_points.loc[label_mask, "umap_x"],
            pseudo_points.loc[label_mask, "umap_y"],
            s=point_sizes,
            alpha=0.55,
            color=palette[class_name],
            label=class_name,
        )
axes[1].set_title(f"Pseudo Labels Projected Into Labeled Space (confidence >= {umap_min_confidence:.2f})")
axes[1].set_xlabel("UMAP-1")
axes[1].set_ylabel("UMAP-2")
axes[1].grid(alpha=0.2)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

umap_figure_path = umap_output_dir / "seed07_pseudolabel_umap.png"
umap_labeled_csv = umap_output_dir / "labeled_reference_umap.csv"
umap_pseudo_csv = umap_output_dir / "pseudo_label_umap.csv"
umap_summary_json = umap_output_dir / "seed07_pseudolabel_umap.summary.json"

fig.savefig(umap_figure_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

labeled_points.to_csv(umap_labeled_csv, index=False)
pseudo_points.to_csv(umap_pseudo_csv, index=False)
umap_summary = {
    "checkpoint": str(checkpoint_path),
    "raw_pickle": str(RAW_PICKLE),
    "min_confidence": float(umap_min_confidence),
    "max_labeled_per_class": int(umap_max_labeled_per_class),
    "max_pseudo_per_class": int(umap_max_pseudo_per_class),
    "n_neighbors": int(umap_n_neighbors),
    "min_dist": float(umap_min_dist),
    "metric": umap_metric,
    "labeled_points": int(len(labeled_points)),
    "pseudo_points": int(len(pseudo_points)),
    "labeled_distribution": labeled_points["true_label"].value_counts().sort_index().to_dict(),
    "pseudo_distribution": pseudo_points["pseudo_label"].value_counts().sort_index().to_dict(),
    "mean_pseudo_confidence": float(pseudo_points["pseudo_label_confidence"].mean()),
    "figure_path": str(umap_figure_path),
    "labeled_csv": str(umap_labeled_csv),
    "pseudo_csv": str(umap_pseudo_csv),
}
umap_summary_json.write_text(json.dumps(umap_summary, indent=2), encoding="utf-8")

display(pd.DataFrame([
    {
        "set": "labeled_reference",
        "points": len(labeled_points),
        "min_confidence": None,
    },
    {
        "set": "pseudo_projection",
        "points": len(pseudo_points),
        "min_confidence": umap_min_confidence,
    },
]))
print("Saved UMAP figure to:", umap_figure_path)
print("Saved labeled UMAP points to:", umap_labeled_csv)
print("Saved pseudo-label UMAP points to:", umap_pseudo_csv)
print("Saved UMAP summary to:", umap_summary_json)
